# 03 · Treatment rooms

*Two ways to find a room, and why only one of them is safe.*

**About 15 minutes.**

> **Not a medical device. Teaching use only.** Everything here talks to a *simulator* running on
> your own machine. The surfaces are synthetic, the clearance maps are plausible rather than
> physical, and nothing in this repository came from a patient. Not affiliated with or endorsed
> by Vision RT.

First, make sure the environment is ready. On your own machine this cell does nothing at all — it only matters on Colab and Binder, where the simulator has to be installed and started from inside the notebook.

In [ ]:
# Running on Google Colab? This installs the workshop and starts the simulator.
# Anywhere else it does nothing: locally you already started it in the other terminal, and on
# Binder it was started for you before this notebook opened.
import os, subprocess, sys, time, urllib.request

if "COLAB_RELEASE_TAG" in os.environ:
    REPO = "https://github.com/tallhamer/SGRT-Scripting-Workshop"
    CHECKOUT = "/content/workshop"

    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    f"maprt-scripting-workshop[notebooks] @ git+{REPO}"], check=True)

    if not os.path.isdir(CHECKOUT):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CHECKOUT], check=True)

    # Work from the checkout's notebooks/ directory. Colab opens a notebook detached from its
    # repository, so every relative path would otherwise be wrong — the capstone reads
    # "../data/MapRT/plans/..." and would look in /data. One chdir makes Colab match a local clone
    # instead of special-casing paths in each notebook.
    os.chdir(f"{CHECKOUT}/notebooks")

    os.environ["MAPRT_EMULATOR_TOKEN"] = "workshop-colab-token"
    os.environ["MAPRT_EMULATOR_URL"] = "http://127.0.0.1:8000"

    # --token matters: without it the simulator mints a random token, while the environment
    # above advertises a different one, and every call fails with InvalidClientToken. The
    # simulator must be given the same secret this notebook is going to send.
    log = open("/content/maprt-emulator.log", "w")
    subprocess.Popen(["maprt-emulator", "--port", "8000",
                      "--token", os.environ["MAPRT_EMULATOR_TOKEN"],
                      "--data-dir", f"{CHECKOUT}/data/MapRT",
                      "--map-seconds", "1"],
                     stdout=log, stderr=subprocess.STDOUT)

    # Poll rather than sleep. A fixed wait is either too short on a slow runtime (the first cell
    # fails for no visible reason) or wasted time on a fast one.
    for _attempt in range(60):
        try:
            urllib.request.urlopen(os.environ["MAPRT_EMULATOR_URL"] + "/emulator/state", timeout=1)
            print("simulator ready")
            break
        except Exception:
            time.sleep(0.5)
    else:
        print("The simulator did not start. See /content/maprt-emulator.log")

print("Python", sys.version.split()[0])

In [ ]:
from maprt_workshop import connect

maprt = connect()          # finds the simulator and your token, and checks it answers
print(maprt.api_url)

## What a treatment room is here

Not a room — a **configuration**. The appliance holds a scan of a couch and a gantry, and that
scanned geometry is what a clearance map is computed against.

One physical linac often has several of them. A different couch top, an accessory mounted, a
different immobilisation setup: each needs its own scanned configuration, and each is a separate
"treatment room" as far as the API is concerned.

That single fact is the whole of this notebook.

## Listing them

`GET /integration/rooms`. No parameters.

In [ ]:
import httpx2

url = maprt.api_url + "/integration/rooms"
headers = dict(maprt.headers)

response = httpx2.get(url, headers=headers)
response.status_code

Same envelope as always. Check `success` before anything else:

In [ ]:
envelope = response.json()
envelope["success"]

`data` is a list. Look at **one** entry first, to learn the shape:

In [ ]:
rooms = envelope["data"]
rooms[0]

Four fields:

| field | what it is |
| --- | --- |
| `id` | a **GUID**. Unique. This is what `GetMap` wants. |
| `machineName` | the treatment machine, max 16 characters. **Not unique.** |
| `name` | the configuration's display name, as Vision RT's own interface shows it. |
| `coordinateSystem` | which couch coordinate convention this room's angles are reported in. |

Now the whole list:

In [ ]:
import pandas as pd

table = pd.DataFrame(rooms)
table

## Look carefully at `machineName`

Two of those rows share one. That is not a mistake in the test data — it is what a real site looks
like, and it is the trap this notebook exists for.

In [ ]:
table["machineName"].value_counts()

And the GUIDs, by contrast:

In [ ]:
print("rooms          :", len(rooms))
print("unique names   :", table["machineName"].nunique())
print("unique GUIDs   :", table["id"].nunique())

## The other way to find a room

`GET /integration/rooms/{machineName}` resolves a *single* room by its machine name. It looks like
the convenient one. It is the one to be careful with.

Start with a name that matches exactly one room:

In [ ]:
response = httpx2.get(maprt.api_url + "/integration/rooms/TrueBeam2", headers=headers)
response.json()

`data` is the GUID, as a bare string. Good. Now a name that matches **nothing**:

In [ ]:
response = httpx2.get(maprt.api_url + "/integration/rooms/NoSuchMachine", headers=headers)
response.json()

**Read that again.** `success` is `true`. There is no error. And `data` is:

```
00000000-0000-0000-0000-000000000000
```

An all-zeros GUID. "No match" is reported as a **successful call with a null answer**, not as a
failure. A client that only checks `success` will carry that GUID straight into a `GetMap`
request and spend a while wondering why nothing works.

So: check for it.

In [ ]:
EMPTY_GUID = "00000000-0000-0000-0000-000000000000"

def find_room(machine_name):
    envelope = httpx2.get(maprt.api_url + f"/integration/rooms/{machine_name}",
                          headers=headers).json()
    if not envelope["success"]:
        raise RuntimeError(f"{envelope['errorCodeType']}: {envelope['errorCode']}")
    room_id = envelope["data"]
    if room_id == EMPTY_GUID:
        raise LookupError(f"no treatment room is configured for machine {machine_name!r}")
    return room_id


find_room("TrueBeam2")

In [ ]:
try:
    find_room("NoSuchMachine")
except LookupError as problem:
    print("caught:", problem)

## And the third case

A name that matches *more than one* room. The appliance cannot pick for you, so it refuses.

In [ ]:
response = httpx2.get(maprt.api_url + "/integration/rooms/TrueBeam", headers=headers)

print("status:", response.status_code)
response.json()

`DuplicateKey`, at HTTP 200 — a negative result, exactly as notebook 02 described. The C# client
raises this as an `AmbiguousMatchException`; over REST it arrives as an error code.

So `GET /rooms/{machineName}` has **three** outcomes and they are all different:

| | `success` | `data` | meaning |
| --- | --- | --- | --- |
| one match | `true` | the GUID | what you wanted |
| no match | `true` | all-zeros GUID | not configured |
| several matches | `false` | `null` | `DuplicateKey` — ambiguous |

Our `find_room` above handles all three. That is three branches for what looks like a one-line
lookup, which is why production code generally does not use this endpoint at all.

## What production code does instead

List the rooms once. Let a human pick. Key on the GUID from then on.

The GUID is unique, it is stable, and it is what `GetMap` and `GetBeamDeliveryStatus` actually
take. The machine name is for showing people.

In [ ]:
rooms = maprt.get_treatment_rooms()          # [(name, id, coordinate_system), ...]

for index, (name, room_id, coordinate_system) in enumerate(rooms):
    print(f"  [{index}] {name:<20} {coordinate_system:<12} {room_id}")

# In an application this is a dropdown. Here, pick one.
chosen_name, ROOM_ID, _coordinate_system = rooms[0]
print(f"\nusing: {chosen_name}  ->  {ROOM_ID}")

## One more field: `coordinateSystem`

Notice the rooms do not all agree. One of them reports `Native`.

In [ ]:
table[["name", "machineName", "coordinateSystem"]]

A room reports the couch coordinate convention Vision RT's own user interface displays it in. The
specification supports six:

| value | |
| --- | --- |
| `Native` | Vision RT's internal convention |
| `IEC_61217` | **what DICOM uses, and what the clearance map comes back in** |
| `IEC_60601_2_1`, `IEC_60601_NW` | other IEC conventions |
| `OBI_60601_1_5`, `OBI_60601_1_6` | vendor variants |

For this workshop everything is IEC 61217 and you can ignore the field. In the clinic you cannot:
the conversions are vendor-specific and live in a separate Vision RT document. If you are
comparing an angle from the API against an angle from a planning system or a console readout,
**check which convention each one is in before you trust the comparison.**

The simulator refuses angles in anything but IEC 61217 rather than silently converting them
wrongly — see notebook 06.

## Your turn

Write `resolve_room(machine_name)` that returns a GUID, and handles all three outcomes with a
*useful* message for each. Then call it for `TrueBeam2`, `Halcyon`, `TrueBeam`, and `Nope`.

What should happen for the ambiguous one? There is no single right answer — think about what a
planner could actually do about it.

In [ ]:
# Your turn.
def resolve_room(machine_name):
    ...

<details><summary>Answer</summary>

Run the next cell.

</details>

In [ ]:
from maprt_workshop import MapRTApiError

def resolve_room(machine_name):
    """A room GUID, or a message a planner can act on."""
    try:
        envelope = maprt.get_treatment_room(machine_name)
    except MapRTApiError as problem:
        if problem.error_code == "DuplicateKey":
            # Ambiguous. Nobody can fix this from the name alone, so fall back to the list and
            # show the alternatives — the planner knows which configuration they meant.
            options = [name for name, _id, _cs in maprt.get_treatment_rooms()]
            raise LookupError(
                f"{machine_name!r} matches several room configurations. "
                f"Pick one by name: {options}") from problem
        raise

    if envelope["data"] == EMPTY_GUID:
        raise LookupError(f"no room is configured for machine {machine_name!r}. "
                          f"Check the name against /integration/rooms.")
    return envelope["data"]


for machine in ("TrueBeam2", "Halcyon", "TrueBeam", "Nope"):
    try:
        print(f"{machine:<12} -> {resolve_room(machine)}")
    except LookupError as problem:
        print(f"{machine:<12} -> {problem}")

---

**Keep `ROOM_ID`.** The next notebooks need it, along with a surface id.

**Next:** [`04_patient_surfaces.ipynb`](04_patient_surfaces.ipynb) — getting a patient's mesh, and
the trap in what the bytes do *not* tell you.